# DS Quest practice: Responsible AI: Fairness, Privacy & Governance

**Fill-in-the-blank version:** each cell starts with a skeleton; replace every `___`. (The blank-page version is `respai-blank.ipynb`.)

Practical questions from the **Responsible AI: Fairness, Privacy & Governance** module. Write your answer in the code cell under each question, run it, then run the **check** cell right below it. The check runs *in this notebook*, on your own variables, and tells you exactly what is right or wrong.

1. Run the first code cell once (it loads the checker and the 12 questions).
2. Work through the questions in any order.
3. `dsq.hint("question-id")` gives a hint; `dsq.solution("question-id")` shows the reference answer (only after you tried 3 times).
4. `dsq.score()` shows where you stand.

Optional: `dsq.connect("http://127.0.0.1:5177", "YOUR-KEY")` (key from the module page in DS Quest) sends solved questions to the app so your profile counts them.

You need Python with `numpy`, `pandas`, `matplotlib` and `seaborn`, e.g. `pip install numpy pandas matplotlib seaborn notebook`.

In [ ]:
# --- DS Quest practice checker (runs in YOUR notebook, on YOUR variables) ---
import json as _json, sys as _sys, urllib.request as _ur

class _DSQ:
    def __init__(self):
        self.qs = {}; self.order = []; self.tries = {}; self.ok = {}; self.url = None; self.key = None; self.module = ''

    def load(self, pack):
        self.module = pack['title']
        for q in pack['questions']:
            self.qs[q['id']] = q; self.order.append(q['id'])
        print(f"DS Quest: {len(self.order)} questions loaded for '{self.module}'. Write your answer, then run the check cell under it.")
        if self._auto(): print('Linked to your DS Quest account: solved questions count in the app.')

    def connect(self, url, key, quiet=False):
        self.url = url.rstrip('/'); self.key = key
        if not quiet: print('Connected: solved questions will be reported to', self.url)

    def _auto(self):
        # DS Quest writes dsq_config.json next to the notebooks when you open them from the app
        import os
        d = os.getcwd()
        for _ in range(4):
            p = os.path.join(d, 'dsq_config.json')
            if os.path.exists(p):
                try:
                    c = _json.load(open(p)); self.connect(c['url'], c['key'], quiet=True); return True
                except Exception:
                    pass
            d = os.path.dirname(d)
        return False

    def _report(self, qid, ok):
        if not self.url: return
        try:
            req = _ur.Request(self.url + '/api/nb/report', data=_json.dumps({'key': self.key, 'qid': qid, 'ok': bool(ok)}).encode(), headers={'Content-Type': 'application/json'})
            _ur.urlopen(req, timeout=4).read()
        except Exception as e:
            print('(could not reach the app:', str(e)[:80] + ')')

    def check(self, qid):
        q = self.qs[qid]
        try:
            ip = get_ipython(); ns = dict(ip.user_ns)
        except NameError:
            ns = dict(_sys.modules['__main__'].__dict__)
        results = []
        def check(cond, msg, hint=''):
            ok = bool(cond); results.append((ok, msg, hint)); return ok
        ns['check'] = check; ns['OUT'] = ''
        for mod, alias in (('numpy', 'np'), ('pandas', 'pd')):
            if alias not in ns:
                try: ns[alias] = __import__(mod)
                except Exception: pass
        if 'plt' not in ns:
            try:
                import matplotlib.pyplot as plt; ns['plt'] = plt
            except Exception: pass
        try:
            exec(compile(q['tests'], '<tests>', 'exec'), ns)
        except BaseException as e:
            results.append((False, 'A check could not finish: ' + type(e).__name__ + ': ' + str(e)[:200], 'Did you create every variable name the task asks for? Did you run your answer cell first?'))
        self.tries[qid] = self.tries.get(qid, 0) + 1
        good = bool(results) and all(r[0] for r in results)
        for ok, msg, hint in results:
            print(('  [PASS] ' if ok else '  [FAIL] ') + msg + ('' if ok or not hint else '\n         hint: ' + hint))
        if good:
            self.ok[qid] = True; print(f"\n  Question solved: {q['title']}  (attempt {self.tries[qid]})")
        else:
            print(f"\n  Not yet - fix your answer cell, run it again, then re-run this check. (attempt {self.tries[qid]})")
        self._report(qid, good)
        return good

    def hint(self, qid):
        hs = self.qs[qid].get('hints') or []
        n = min(self.tries.get(qid, 0), len(hs)) if hs else 0
        print(('hint: ' + hs[max(0, n - 1)]) if hs else 'No hint for this one - re-read the question.')

    def solution(self, qid):
        if self.tries.get(qid, 0) < 3 and not self.ok.get(qid):
            print('Try the check at least 3 times first - the struggle is where the learning happens.'); return
        print(self.qs[qid]['solution'])

    def score(self):
        n = len(self.order); s = sum(1 for q in self.order if self.ok.get(q))
        print(f"Solved {s} of {n}.")
        for q in self.order:
            print(('  [x] ' if self.ok.get(q) else '  [ ] ') + q + ' - ' + self.qs[q]['title'] + ('' if self.ok.get(q) else f"  (tries: {self.tries.get(q, 0)})"))

dsq = _DSQ()


dsq.load({
 "module": "respai",
 "title": "Responsible AI: Fairness, Privacy & Governance",
 "questions": [
  {
   "id": "ra-1-1",
   "title": "Selection rates and the four-fifths rule",
   "tests": "p = [1, 1, 1, 0, 1, 0, 0, 0]; g = ['a', 'a', 'a', 'a', 'b', 'b', 'b', 'b']\nr = selection_rates(p, g)\ncheck(r == {'a': 0.75, 'b': 0.25}, 'group a is selected 75% of the time, group b 25%', 'positives / members per group')\ncheck(abs(dp_gap(r) - 0.5) < 1e-12 and abs(impact_ratio(r) - 1/3) < 1e-12, 'a gap of 0.5; the lower rate is a third of the higher, far below 0.8', 'max - min and min / max')\ncheck(impact_ratio({'a': 0.0, 'b': 0.0}) == 1.0 and impact_ratio({'a': 0.5, 'b': 0.45}) >= 0.8, 'nobody selected is 1.0; 0.45 vs 0.5 passes the four-fifths rule', 'guard max = 0')",
   "hints": [
    "Count members and positives per group."
   ],
   "solution": "def selection_rates(preds, groups):\n    tot, pos = {}, {}\n    for p, g in zip(preds, groups):\n        tot[g] = tot.get(g, 0) + 1\n        pos[g] = pos.get(g, 0) + p\n    return {g: pos[g] / tot[g] for g in tot}\n\ndef dp_gap(rates):\n    return max(rates.values()) - min(rates.values())\n\ndef impact_ratio(rates):\n    hi = max(rates.values())\n    return 1.0 if hi == 0 else min(rates.values()) / hi\n\np = [1, 1, 1, 0, 1, 0, 0, 0]\ng = [\"a\", \"a\", \"a\", \"a\", \"b\", \"b\", \"b\", \"b\"]\nr = selection_rates(p, g)\nprint(r, dp_gap(r), impact_ratio(r))"
  },
  {
   "id": "ra-1-2",
   "title": "Equal opportunity",
   "tests": "yt = [1, 1, 1, 1, 0, 1, 1, 1, 1, 0]; yp = [1, 1, 1, 0, 0, 1, 0, 0, 0, 0]; g = ['a'] * 5 + ['b'] * 5\nt = tpr_by_group(yt, yp, g)\ncheck(t == {'a': 0.75, 'b': 0.25}, 'group a: 3 of 4 qualified people found; group b: 1 of 4', 'hits / positives per group')\ncheck(abs(eo_gap(t) - 0.5) < 1e-12, 'a 50-point equal-opportunity gap', 'max - min')\ncheck(tpr_by_group([0, 0], [1, 0], ['a', 'a']) == {}, 'a group with no qualified members has no TPR', 'skip groups without positives')",
   "hints": [
    "Only look at rows where the truth is 1."
   ],
   "solution": "def tpr_by_group(y_true, y_pred, groups):\n    pos, hit = {}, {}\n    for yt, yp, g in zip(y_true, y_pred, groups):\n        if yt == 1:\n            pos[g] = pos.get(g, 0) + 1\n            hit[g] = hit.get(g, 0) + (1 if yp == 1 else 0)\n    return {g: hit[g] / pos[g] for g in pos}\n\ndef eo_gap(tprs):\n    return max(tprs.values()) - min(tprs.values())\n\nyt = [1, 1, 1, 1, 0, 1, 1, 1, 1, 0]\nyp = [1, 1, 1, 0, 0, 1, 0, 0, 0, 0]\ng = [\"a\"] * 5 + [\"b\"] * 5\nt = tpr_by_group(yt, yp, g)\nprint(t, eo_gap(t))"
  },
  {
   "id": "ra-1-3",
   "title": "Calibration by group",
   "tests": "s = [0.72, 0.75, 0.71, 0.74, 0.73, 0.70, 0.76, 0.72]; y = [1, 1, 1, 0, 1, 0, 0, 0]; g = ['a'] * 4 + ['b'] * 4\nout = calibration_by_group(s, y, g, 0.7, 0.8)\ncheck(out == {'a': 0.75, 'b': 0.25}, 'at the same score of about 0.73, group a is right 75% of the time and group b only 25%', 'the hit rate inside the band')\ncheck(calibration_by_group(s, y, g, 0.9, 1.0) == {'a': None, 'b': None}, 'nobody has a score that high: None for every group', 'empty bands give None')\ncheck(calibration_by_group([0.5, 0.5], [1, 0], ['a', 'b'], 0.5, 0.6) == {'a': 1.0, 'b': 0.0}, 'the lower edge is included', 'lo <= score < hi')",
   "hints": [
    "Register every group first so empty bands give None."
   ],
   "solution": "def calibration_by_group(scores, y_true, groups, lo, hi):\n    n, pos = {}, {}\n    for s, y, g in zip(scores, y_true, groups):\n        n.setdefault(g, 0); pos.setdefault(g, 0)\n        if lo <= s < hi:\n            n[g] += 1\n            pos[g] += y\n    return {g: (pos[g] / n[g] if n[g] else None) for g in n}\n\ns = [0.72, 0.75, 0.71, 0.74, 0.73, 0.70, 0.76, 0.72]\ny = [1, 1, 1, 0, 1, 0, 0, 0]\ng = [\"a\"] * 4 + [\"b\"] * 4\nprint(calibration_by_group(s, y, g, 0.7, 0.8))"
  },
  {
   "id": "ra-2-1",
   "title": "A threshold per group",
   "tests": "a = [0.9, 0.8, 0.6, 0.4, 0.2]\ncheck(threshold_for_rate(a, 0.4) == 0.8, 'to select the top 40% (2 of 5) the threshold is the second-highest score', 'k = round(rate * n)')\ncheck(threshold_for_rate(a, 0.0) == float('inf') and threshold_for_rate(a, 1.0) == 0.2, 'rate 0 selects nobody; rate 1 selects everybody', 'edge cases')\ncheck(apply_thresholds([0.7, 0.7], ['a', 'b'], {'a': 0.75, 'b': 0.6}) == [0, 1], 'the same score is a no for group a and a yes for group b', 'compare with the group\\'s own threshold')",
   "hints": [
    "sorted(scores, reverse=True)[k - 1] is the k-th highest."
   ],
   "solution": "def threshold_for_rate(scores, rate):\n    k = round(rate * len(scores))\n    if k == 0:\n        return float(\"inf\")\n    return sorted(scores, reverse=True)[k - 1]\n\ndef apply_thresholds(scores, groups, th):\n    return [1 if s >= th[g] else 0 for s, g in zip(scores, groups)]\n\na = [0.9, 0.8, 0.6, 0.4, 0.2]\nprint(threshold_for_rate(a, 0.4), apply_thresholds([0.7, 0.7], [\"a\", \"b\"], {\"a\": 0.75, \"b\": 0.6}))"
  },
  {
   "id": "ra-2-2",
   "title": "Reweighing",
   "tests": "g = ['a', 'a', 'a', 'a', 'b', 'b', 'b', 'b']; y = [1, 1, 1, 0, 1, 0, 0, 0]\nw = reweigh(g, y)\ncheck(abs(w[('a', 1)] - (0.5 * 0.5 / 0.375)) < 1e-9, 'group a with a positive label is common: weight 2/3', 'P(g) P(y) / P(g, y)')\ncheck(abs(w[('b', 1)] - 2.0) < 1e-9, 'group b with a positive label is rare: weight 2', 'rare combinations count more')\ncheck(abs(sum(w[k] * sum(1 for gg, yy in zip(g, y) if (gg, yy) == k) for k in w) - 8) < 1e-9, 'the weights keep the total data size', 'weights sum to n')",
   "hints": [
    "Count groups, labels and (group, label) pairs, then divide by n."
   ],
   "solution": "def reweigh(groups, labels):\n    n = len(groups)\n    pg, pl, pj = {}, {}, {}\n    for g, y in zip(groups, labels):\n        pg[g] = pg.get(g, 0) + 1\n        pl[y] = pl.get(y, 0) + 1\n        pj[(g, y)] = pj.get((g, y), 0) + 1\n    return {(g, y): (pg[g] / n) * (pl[y] / n) / (c / n) for (g, y), c in pj.items()}\n\ng = [\"a\", \"a\", \"a\", \"a\", \"b\", \"b\", \"b\", \"b\"]\ny = [1, 1, 1, 0, 1, 0, 0, 0]\nprint(reweigh(g, y))"
  },
  {
   "id": "ra-2-3",
   "title": "Why you cannot have everything",
   "tests": "check(abs(precision_from(0.5, 0.8, 0.2) - 0.8) < 1e-12 and abs(precision_from(0.2, 0.8, 0.2) - 0.5) < 1e-12, 'same error rates, different base rates: 80% precision versus 50%', 'tp / (tp + fp)')\ncheck(abs(precision_gap(0.5, 0.2, 0.8, 0.2) - 0.3) < 1e-12, 'a 30-point precision gap although TPR and FPR are identical', 'difference of the two precisions')\ncheck(precision_gap(0.3, 0.3, 0.9, 0.1) == 0.0, 'equal base rates give no conflict', 'the gap needs different base rates')",
   "hints": [
    "Precision = true positives / all predicted positives."
   ],
   "solution": "def precision_from(base_rate, tpr, fpr):\n    b = base_rate\n    return tpr * b / (tpr * b + fpr * (1 - b))\n\ndef precision_gap(b1, b2, tpr, fpr):\n    return abs(precision_from(b1, tpr, fpr) - precision_from(b2, tpr, fpr))\n\nprint(precision_from(0.5, 0.8, 0.2), precision_from(0.2, 0.8, 0.2), precision_gap(0.5, 0.2, 0.8, 0.2))"
  },
  {
   "id": "ra-3-1",
   "title": "How anonymous is this table?",
   "tests": "rows = [{'zip': '380', 'age': '30-39'}] * 3 + [{'zip': '390', 'age': '40-49'}] * 2 + [{'zip': '390', 'age': '30-39'}]\ncheck(k_anonymity(rows, ['zip', 'age']) == 1, 'one person is unique on (zip, age): the table is only 1-anonymous', 'the smallest group')\ncheck(violations(rows, ['zip', 'age'], 2) == [('390', '30-39')], 'that one combination breaks 2-anonymity', 'groups smaller than k')\ncheck(k_anonymity(rows, ['zip']) == 3 and violations(rows, ['zip'], 3) == [], 'using only the zip column, everyone hides in a group of 3 or more', 'fewer quasi-identifiers, bigger groups')",
   "hints": [
    "Use a tuple of the quasi-identifier values as the dictionary key."
   ],
   "solution": "def k_anonymity(rows, quasi):\n    counts = {}\n    for r in rows:\n        key = tuple(r[c] for c in quasi)\n        counts[key] = counts.get(key, 0) + 1\n    return min(counts.values())\n\ndef violations(rows, quasi, k):\n    counts = {}\n    for r in rows:\n        key = tuple(r[c] for c in quasi)\n        counts[key] = counts.get(key, 0) + 1\n    return sorted(key for key, n in counts.items() if n < k)\n\nrows = [{\"zip\": \"380\", \"age\": \"30-39\"}] * 3 + [{\"zip\": \"390\", \"age\": \"40-49\"}] * 2 + [{\"zip\": \"390\", \"age\": \"30-39\"}]\nprint(k_anonymity(rows, [\"zip\", \"age\"]), violations(rows, [\"zip\", \"age\"], 2))"
  },
  {
   "id": "ra-3-2",
   "title": "A differentially private count",
   "tests": "import random, statistics\nrng = random.Random(1)\nxs = [laplace_noise(2.0, rng) for _ in range(20000)]\ncheck(abs(statistics.mean(xs)) < 0.1 and abs(statistics.pstdev(xs) - 2.0 * 2 ** 0.5) < 0.15, 'noise is centred on 0 with standard deviation scale x sqrt(2)', 'a Laplace distribution')\nerr_small = statistics.mean(abs(dp_count(100, 0.1, s) - 100) for s in range(500))\nerr_big = statistics.mean(abs(dp_count(100, 2.0, s) - 100) for s in range(500))\ncheck(err_small > 5 * err_big and abs(err_small - 10) < 2, 'epsilon = 0.1 means about 10 counts of noise; epsilon = 2 about 0.5', 'scale = 1 / epsilon')\ncheck(dp_count(100, 1.0, 3) == dp_count(100, 1.0, 3), 'the same seed gives the same answer (reproducible)', 'build the generator inside')",
   "hints": [
    "This is the inverse-CDF way to draw a Laplace sample."
   ],
   "solution": "import math, random\n\ndef laplace_noise(scale, rng):\n    u = rng.random() - 0.5\n    sign = 1 if u >= 0 else -1\n    return -scale * sign * math.log(max(1 - 2 * abs(u), 1e-12))\n\ndef dp_count(true_count, epsilon, seed):\n    rng = random.Random(seed)\n    return true_count + laplace_noise(1 / epsilon, rng)\n\nprint(dp_count(100, 1.0, 0))"
  },
  {
   "id": "ra-3-3",
   "title": "Randomised response",
   "tests": "import random\nrng = random.Random(0)\ntruth = [1 if rng.random() < 0.3 else 0 for _ in range(20000)]\nrep = randomized_response(truth, 0.75, 1)\ncheck(len(rep) == len(truth) and set(rep) <= {0, 1}, 'one reported bit per person', 'keep or flip')\ncheck(abs(sum(rep) / len(rep) - (0.3 * 0.75 + 0.7 * 0.25)) < 0.02, 'the raw reports are distorted: about 40% say yes, not 30%', 'observed = true p + (1 - true)(1 - p)')\ncheck(abs(estimate_rate(rep, 0.75) - sum(truth) / len(truth)) < 0.03, 'but the corrected estimate recovers the true rate (about 30%)', '(observed - (1 - p)) / (2p - 1)')",
   "hints": [
    "A flipped bit is 1 - b."
   ],
   "solution": "import random\n\ndef randomized_response(bits, p, seed):\n    rng = random.Random(seed)\n    return [b if rng.random() < p else 1 - b for b in bits]\n\ndef estimate_rate(reports, p):\n    return (sum(reports) / len(reports) - (1 - p)) / (2 * p - 1)\n\nrng = random.Random(0)\ntruth = [1 if rng.random() < 0.3 else 0 for _ in range(20000)]\nrep = randomized_response(truth, 0.75, 1)\nprint(sum(rep) / len(rep), estimate_rate(rep, 0.75))"
  },
  {
   "id": "ra-4-1",
   "title": "Classify the risk tier",
   "tests": "check(risk_tier({'social_scoring': True, 'domain': 'hiring'}) == 'unacceptable', 'a prohibited practice wins over everything else', 'check the top tier first')\ncheck(risk_tier({'domain': 'credit', 'interacts_with_people': True}) == 'high' and risk_tier({'domain': 'health'}) == 'high', 'a high-risk domain is high even if it also chats with people', 'domain before interaction')\ncheck(risk_tier({'generates_content': True}) == 'limited' and risk_tier({'domain': 'games'}) == 'minimal' and risk_tier({}) == 'minimal', 'chatbots are limited; a game AI is minimal', 'the defaults')",
   "hints": [
    "Check from the most serious tier down."
   ],
   "solution": "HIGH_DOMAINS = {\"hiring\", \"credit\", \"education\", \"health\", \"law_enforcement\", \"critical_infrastructure\"}\n\ndef risk_tier(system):\n    if system.get(\"social_scoring\") or system.get(\"realtime_public_biometrics\"):\n        return \"unacceptable\"\n    if system.get(\"domain\") in HIGH_DOMAINS:\n        return \"high\"\n    if system.get(\"interacts_with_people\") or system.get(\"generates_content\"):\n        return \"limited\"\n    return \"minimal\"\n\nprint(risk_tier({\"domain\": \"hiring\"}), risk_tier({\"interacts_with_people\": True}))"
  },
  {
   "id": "ra-4-2",
   "title": "Is the model card complete?",
   "tests": "c = {'intended_use': 'triage', 'owner': 'ml-team', 'limitations': ''}\ncheck(card_missing(c) == ['limitations', 'metrics_by_group', 'out_of_scope', 'training_data'], 'an empty limitations field counts as missing', 'missing or empty')\ncheck(abs(completeness(c) - 2/6) < 1e-12, 'two of six required fields are filled in', '1 - missing / required')\nfull = {f: 'x' for f in REQUIRED}\ncheck(card_missing(full) == [] and completeness(full) == 1.0, 'a full card is 100% complete', 'nothing missing')",
   "hints": [
    "not card.get(field) is True for missing, None and empty values."
   ],
   "solution": "REQUIRED = [\"intended_use\", \"out_of_scope\", \"training_data\", \"metrics_by_group\", \"limitations\", \"owner\"]\n\ndef card_missing(card):\n    return sorted(f for f in REQUIRED if not card.get(f))\n\ndef completeness(card):\n    return 1 - len(card_missing(card)) / len(REQUIRED)\n\nprint(card_missing({\"intended_use\": \"triage\", \"owner\": \"ml-team\", \"limitations\": \"\"}))"
  },
  {
   "id": "ra-4-3",
   "title": "Red-team results",
   "tests": "res = [('jailbreak', True), ('jailbreak', False), ('injection', True), ('injection', True), ('bias', False), ('bias', False)]\noverall, by = asr_report(res)\ncheck(abs(overall - 0.5) < 1e-12 and by == {'jailbreak': 0.5, 'injection': 1.0, 'bias': 0.0}, 'half of all attacks worked; injection always did', 'successes / attempts, overall and per category')\ncheck(worst(res) == 'injection', 'injection is the weakest area', 'the highest success rate')\ncheck(worst([('b', True), ('a', True)]) == 'a', 'ties go to the alphabetically first category', 'sort key (-rate, name)')",
   "hints": [
    "Sort by (-rate, name) to break ties alphabetically."
   ],
   "solution": "def asr_report(results):\n    tot, ok = {}, {}\n    for cat, success in results:\n        tot[cat] = tot.get(cat, 0) + 1\n        ok[cat] = ok.get(cat, 0) + (1 if success else 0)\n    overall = sum(ok.values()) / len(results)\n    return overall, {c: ok[c] / tot[c] for c in tot}\n\ndef worst(results):\n    _, by = asr_report(results)\n    return sorted(by, key=lambda c: (-by[c], c))[0]\n\nres = [(\"jailbreak\", True), (\"jailbreak\", False), (\"injection\", True), (\"injection\", True), (\"bias\", False), (\"bias\", False)]\nprint(asr_report(res), worst(res))"
  }
 ]
})


## 1. Selection rates and the four-fifths rule  `ra-1-1`
*(from the lesson: Measuring fairness between groups)*

Write `selection_rates(preds, groups)` returning a dict `group -> share of that group with prediction 1`, then `dp_gap(rates)` = highest minus lowest rate, and `impact_ratio(rates)` = lowest / highest (1.0 if the highest is 0). A system passes the four-fifths rule when the ratio is at least 0.8.

In [ ]:
def selection_rates(preds, groups):
    ...

def dp_gap(rates):
    ...

def impact_ratio(rates):
    ...

p = [1, 1, 1, 0, 1, 0, 0, 0]
g = ["a", "a", "a", "a", "b", "b", "b", "b"]
r = selection_rates(p, g)
print(r, dp_gap(r), impact_ratio(r))

In [ ]:
dsq.check("ra-1-1")

## 2. Equal opportunity  `ra-1-2`
*(from the lesson: Measuring fairness between groups)*

Write `tpr_by_group(y_true, y_pred, groups)`: for each group the share of its **truly positive** members that were predicted positive (skip groups with no positives). Then `eo_gap(tprs)` = highest minus lowest.

In [ ]:
def tpr_by_group(y_true, y_pred, groups):
    ...

def eo_gap(tprs):
    ...

yt = [1, 1, 1, 1, 0, 1, 1, 1, 1, 0]
yp = [1, 1, 1, 0, 0, 1, 0, 0, 0, 0]
g = ["a"] * 5 + ["b"] * 5
t = tpr_by_group(yt, yp, g)
print(t, eo_gap(t))

In [ ]:
dsq.check("ra-1-2")

## 3. Calibration by group  `ra-1-3`
*(from the lesson: Measuring fairness between groups)*

Write `calibration_by_group(scores, y_true, groups, lo, hi)`: for each group take only the rows with `lo <= score < hi` and return the share of those rows with `y_true == 1`. A group with no rows in the band gets `None`.

In [ ]:
def calibration_by_group(scores, y_true, groups, lo, hi):
    ...

s = [0.72, 0.75, 0.71, 0.74, 0.73, 0.70, 0.76, 0.72]
y = [1, 1, 1, 0, 1, 0, 0, 0]
g = ["a"] * 4 + ["b"] * 4
print(calibration_by_group(s, y, g, 0.7, 0.8))

In [ ]:
dsq.check("ra-1-3")

## 4. A threshold per group  `ra-2-1`
*(from the lesson: Mitigation and the limits of fairness)*

Write `threshold_for_rate(scores, rate)`: sort the scores from high to low, let `k = round(rate * len(scores))` and return the k-th highest score (so that `score >= threshold` selects k people); return `float("inf")` if `k == 0`. Then `apply_thresholds(scores, groups, th)` where `th` is a dict `group -> threshold` returns a list of 0/1 decisions.

In [ ]:
def threshold_for_rate(scores, rate):
    ...

def apply_thresholds(scores, groups, th):
    ...

a = [0.9, 0.8, 0.6, 0.4, 0.2]
print(threshold_for_rate(a, 0.4), apply_thresholds([0.7, 0.7], ["a", "b"], {"a": 0.75, "b": 0.6}))

In [ ]:
dsq.check("ra-2-1")

## 5. Reweighing  `ra-2-2`
*(from the lesson: Mitigation and the limits of fairness)*

Write `reweigh(groups, labels)` returning a dict `(group, label) -> weight` with `weight = P(group) * P(label) / P(group, label)`, where the probabilities are shares of the whole data set. Only combinations that occur appear in the dict.

In [ ]:
def reweigh(groups, labels):
    ...

g = ["a", "a", "a", "a", "b", "b", "b", "b"]
y = [1, 1, 1, 0, 1, 0, 0, 0]
print(reweigh(g, y))

In [ ]:
dsq.check("ra-2-2")

## 6. Why you cannot have everything  `ra-2-3`
*(from the lesson: Mitigation and the limits of fairness)*

Write `precision_from(base_rate, tpr, fpr)` = `tpr * b / (tpr * b + fpr * (1 - b))`. Then `precision_gap(b1, b2, tpr, fpr)` = the absolute difference between the precision of the two groups when both have the **same** TPR and FPR but base rates `b1` and `b2`.

In [ ]:
def precision_from(base_rate, tpr, fpr):
    ...

def precision_gap(b1, b2, tpr, fpr):
    ...

print(precision_from(0.5, 0.8, 0.2), precision_from(0.2, 0.8, 0.2), precision_gap(0.5, 0.2, 0.8, 0.2))

In [ ]:
dsq.check("ra-2-3")

## 7. How anonymous is this table?  `ra-3-1`
*(from the lesson: Privacy: k-anonymity, noise and randomised response)*

`rows` is a list of dicts. Write `k_anonymity(rows, quasi)`: the size of the **smallest** group of rows sharing the same values of the columns in `quasi` (a list of column names). Then `violations(rows, quasi, k)`: the sorted list of value-tuples whose group has fewer than `k` rows.

In [ ]:
def k_anonymity(rows, quasi):
    ...

def violations(rows, quasi, k):
    ...

rows = [{"zip": "380", "age": "30-39"}] * 3 + [{"zip": "390", "age": "40-49"}] * 2 + [{"zip": "390", "age": "30-39"}]
print(k_anonymity(rows, ["zip", "age"]), violations(rows, ["zip", "age"], 2))

In [ ]:
dsq.check("ra-3-1")

## 8. A differentially private count  `ra-3-2`
*(from the lesson: Privacy: k-anonymity, noise and randomised response)*

Write `laplace_noise(scale, rng)`: with `u = rng.random() - 0.5`, return `-scale * sign(u) * ln(1 - 2*|u|)` (guard the logarithm with `max(1 - 2*abs(u), 1e-12)`; treat `u == 0` as sign 1). Then `dp_count(true_count, epsilon, seed)`: with `rng = random.Random(seed)` return `true_count + laplace_noise(1 / epsilon, rng)`.

In [ ]:
import math, random

def laplace_noise(scale, rng):
    ...

def dp_count(true_count, epsilon, seed):
    ...

print(dp_count(100, 1.0, 0))

In [ ]:
dsq.check("ra-3-2")

## 9. Randomised response  `ra-3-3`
*(from the lesson: Privacy: k-anonymity, noise and randomised response)*

Write `randomized_response(bits, p, seed)`: with `rng = random.Random(seed)`, keep each bit with probability `p` (`rng.random() < p`) and flip it otherwise; return the list of reported bits. Then `estimate_rate(reports, p)` = `(mean(reports) - (1 - p)) / (2p - 1)`.

In [ ]:
import random

def randomized_response(bits, p, seed):
    ...

def estimate_rate(reports, p):
    ...

rng = random.Random(0)
truth = [1 if rng.random() < 0.3 else 0 for _ in range(20000)]
rep = randomized_response(truth, 0.75, 1)
print(sum(rep) / len(rep), estimate_rate(rep, 0.75))

In [ ]:
dsq.check("ra-3-3")

## 10. Classify the risk tier  `ra-4-1`
*(from the lesson: Governance, risk tiers and red-teaming)*

Write `risk_tier(system)` where `system` is a dict with optional keys: `social_scoring` and `realtime_public_biometrics` (booleans), `domain` (a string), `interacts_with_people` and `generates_content` (booleans). Return `"unacceptable"` if either of the first two is true; else `"high"` if `domain` is one of `HIGH_DOMAINS`; else `"limited"` if it interacts with people or generates content; else `"minimal"`.

In [ ]:
HIGH_DOMAINS = {"hiring", "credit", "education", "health", "law_enforcement", "critical_infrastructure"}

def risk_tier(system):
    ...

print(risk_tier({"domain": "hiring"}), risk_tier({"interacts_with_people": True}))

In [ ]:
dsq.check("ra-4-1")

## 11. Is the model card complete?  `ra-4-2`
*(from the lesson: Governance, risk tiers and red-teaming)*

Write `card_missing(card)`: the sorted list of names in `REQUIRED` that are absent from the dict `card` or whose value is empty (empty string, empty list/dict, or `None`). Then `completeness(card)`: the share of required fields present, between 0 and 1.

In [ ]:
REQUIRED = ["intended_use", "out_of_scope", "training_data", "metrics_by_group", "limitations", "owner"]

def card_missing(card):
    ...

def completeness(card):
    ...

print(card_missing({"intended_use": "triage", "owner": "ml-team", "limitations": ""}))

In [ ]:
dsq.check("ra-4-2")

## 12. Red-team results  `ra-4-3`
*(from the lesson: Governance, risk tiers and red-teaming)*

`results` is a list of `(category, success)` pairs: one entry per attack attempt, `success` True if the attack worked. Write `asr_report(results)` returning `(overall_rate, by_category)` where `by_category` maps each category to its attack success rate, and `worst(results)`: the category with the highest rate (the alphabetically first on ties).

In [ ]:
def asr_report(results):
    ...

def worst(results):
    ...

res = [("jailbreak", True), ("jailbreak", False), ("injection", True), ("injection", True), ("bias", False), ("bias", False)]
print(asr_report(res), worst(res))

In [ ]:
dsq.check("ra-4-3")

## Done?

In [ ]:
dsq.score()